In [1]:
import pandas as pd
import numpy as np

In [2]:
        
data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_GARNETS_unprocessed.xlsx",header=0,index_col=0)
# data = pd.concat([data,data1],axis=0)
oxide = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)
oxlist = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5"]

def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    oxlist1 = oxlist.copy()
    oxide = oxide.T[oxlist1].iloc[0, :].to_numpy()
    data = normalize(data)
    [r, c] = data.shape
    data_f = np.empty((r, c))
    for i in range(0, c):
        data_f[:, i] = data[:, i] / oxide[i]
    data
    data_f = normalize(data_f)
    return(data_f.round(2))


def normalize(data):
    [r, c] = data.shape
    a = data.sum(axis=1).reshape((len(data), 1))
    data_formatted = ((data*100)/a).round(1)
    return(data_formatted)


def cat_calc(data1,oxide_list):
    total = data1.columns.get_loc("Total")
    o_no = data1.loc[:,"Oxygen_no"]
    data = data1.iloc[:,:total].copy()
    oxide = oxide_list.loc[data.columns]
    data = data.div(oxide['Mol. Wt.'].values,axis=1).round(3)
    data = data.mul(oxide['O_no'].values,axis=1).round(3)
    norm = o_no.div(data.sum(axis=1)).round(3)
    data = data.mul(norm.values,axis=0).round(3)
    data = data.mul(oxide['Cat_per_o'].values,axis=1).round(3)
    total = data.sum(axis=1).round(3)
    data.columns = oxide['Cation'].values
    data['Cation_Total'] = total
    return data.round(3)


In [3]:
data2 = data[['SIO2(WT%)','TIO2(WT%)','AL2O3(WT%)','CR2O3(WT%)','FE2O3T(WT%)', 'FE2O3(WT%)', 'FEOT(WT%)','FEO(WT%)', 'MNO(WT%)','MGO(WT%)','CAO(WT%)', 'NA2O(WT%)','K2O(WT%)','P2O5(WT%)','MINERAL']]

In [4]:
data2.MINERAL.unique()

array(['GARNET', 'GROSSULAR', 'ANDRADITE', 'MELANITE', 'HYDROGARNET',
       'PYROPE', 'ALMANDINE', 'SCHORLOMITE', 'HIBSCHITE', nan],
      dtype=object)

In [5]:
len(data2)

54021

In [6]:
data_cleaned = data2.loc[(~data2['SIO2(WT%)'].isna()),:]
# data_px = data_cleaned.loc[(data_cleaned['MINERAL']=="ILMENITE"),:]
data_px = data_cleaned.copy()
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px

,SiO2,TiO2,Al2O3,Cr2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,,
[331] FREZZOTTI M.-L. (1992),41.22,0.16,23.79,0.22,NaN,0.36,18.09,4.85,NaN,NaN,NaN,GARNET
[331] FREZZOTTI M.-L. (1992),40.69,0.38,23.82,0.17,NaN,0.47,17.76,4.87,NaN,NaN,NaN,GARNET
[331] FREZZOTTI M.-L. (1992),41.19,NaN,23.69,0.62,NaN,0.31,17.89,5.11,NaN,NaN,NaN,GARNET
[333] SEN G. (1991),41.13,0.17,23.14,0.09,10.44,0.3,19.13,5.12,NaN,NaN,NaN,GARNET
[333] SEN G. (1991),42.7,0.01,21.77,0,10.75,NaN,19.01,5.06,NaN,NaN,NaN,GARNET
...,...,...,...,...,...,...,...,...,...,...,...,...
[26146] DASGUPTA A. (2022),37.083,0.02,20.018,0,30.908,2.691,0.564,8.542,0.008,0,NaN,GARNET
[26146] DASGUPTA A. (2022),37.584,0.037,20.527,0,31.243,1.283,0.65,9.383,0,0.005,NaN,GARNET
[26146] DASGUPTA A. (2022),37.225,0.02,20.435,0,30.616,1.175,0.563,9.467,0.006,0,NaN,GARNET


In [7]:
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>99) & (total<101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data
data_px2

,SiO2,TiO2,Al2O3,Cr2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
CITATION,,,,,,,,,,,
[333] SEN G. (1991),42.2,0.0,14.0,0.0,8.9,0.0,29.2,5.7,0.0,0.0,0.0
[333] SEN G. (1991),43.5,0.0,13.1,0.0,9.1,0.0,28.8,5.5,0.0,0.0,0.0
[337] GARCIA M. O. (1987),42.3,0.0,14.2,0.0,11.5,0.0,26.5,5.5,0.0,0.0,0.0
[359] SEN G. (1988),41.8,0.0,13.9,0.0,10.1,0.0,28.4,5.7,0.0,0.0,0.0
[359] SEN G. (1988),43.2,0.0,14.0,0.0,9.9,0.0,27.3,5.6,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
[26146] DASGUPTA A. (2022),43.1,0.0,13.7,0.0,30.0,2.6,0.0,10.6,0.0,0.0,0.0
[26146] DASGUPTA A. (2022),43.8,0.0,14.1,0.0,30.4,0.0,0.0,11.7,0.0,0.0,0.0
[26146] DASGUPTA A. (2022),43.8,0.0,14.2,0.0,30.1,0.0,0.0,11.9,0.0,0.0,0.0


In [8]:
non_essential_sum = data_px2[["TiO2","Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<3]
# m = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
# data_px2 = data_px2[ (m >= 49) & (m <= 51)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
m = data_px2[["FeO","MnO","MgO","CaO"]].sum(axis=1)
data_px2 = data_px2[(data_px2['SiO2']<=43) & (data_px2['SiO2']>=40) & (data_px2['Al2O3']>=13) & (data_px2['Al2O3']<=15) & (m >=40) & (m <= 43)]
data_px2.loc[:,'Mineral'] = "Grt"
data_px2.loc[:,'Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
[3728] NEAL C. R. (1988),43.0,0.0,13.9,7.8,0.0,29.9,5.3,0.0,0.0,0.0,Grt
[3728] NEAL C. R. (1988),42.7,0.0,14.5,7.5,0.0,29.9,5.3,0.0,0.0,0.0,Grt
[5641] SCHAAF P. (1994),42.9,0.0,14.2,19.2,0.0,15.9,7.7,0.0,0.0,0.0,Grt
[5641] SCHAAF P. (1994),42.8,0.0,14.2,19.4,0.0,16.2,7.4,0.0,0.0,0.0,Grt
[6621] DAY R. A. (1992),43.0,0.0,14.0,28.9,0.0,5.9,8.1,0.0,0.0,0.0,Grt
...,...,...,...,...,...,...,...,...,...,...,...
[26108] ZHANG XIUZHENG (2022),42.8,0.0,14.2,24.9,0.0,10.6,7.5,0.0,0.0,0.0,Grt
[26109] URANN B. M. (2022),42.7,0.0,14.3,19.4,0.0,13.7,9.9,0.0,0.0,0.0,Grt
[26109] URANN B. M. (2022),42.9,0.0,14.2,18.8,0.0,12.9,11.2,0.0,0.0,0.0,Grt


In [9]:
data_px2.to_excel("/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/Grt_processed_mol.xlsx")

In [10]:
data_px2.sort_values("SiO2",ascending=True)

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
[17376] KOBUSSEN A. F. (2008),41.4,0.0,15.7,6.1,0.0,30.1,6.8,0.0,0.0,0.0,Grt
[17376] KOBUSSEN A. F. (2008),41.4,0.0,16.1,6.3,0.0,30.3,5.9,0.0,0.0,0.0,Grt
[17376] KOBUSSEN A. F. (2008),41.4,0.0,15.9,5.7,0.0,31.0,6.0,0.0,0.0,0.0,Grt
[17376] KOBUSSEN A. F. (2008),41.5,0.0,16.0,6.3,0.0,30.3,5.8,0.0,0.0,0.0,Grt
[17376] KOBUSSEN A. F. (2008),41.5,0.0,15.6,6.1,0.0,31.5,5.4,0.0,0.0,0.0,Grt
...,...,...,...,...,...,...,...,...,...,...,...
[17376] KOBUSSEN A. F. (2008),43.0,0.0,14.3,6.2,0.0,31.4,5.2,0.0,0.0,0.0,Grt
[25597] ZHOU YANYAN (2015),43.0,0.0,14.4,28.7,0.0,5.4,8.5,0.0,0.0,0.0,Grt
[17376] KOBUSSEN A. F. (2008),43.0,0.0,14.1,5.2,0.0,33.2,4.5,0.0,0.0,0.0,Grt


In [11]:
data_cleaned = data2.loc[~data2['FEO(WT%)'].isna(),:]
# data_px = data_cleaned.loc[(data_cleaned['MINERAL']=="ILMENITE"),:]
data_px = data_cleaned.copy()
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>98) & (total<101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data
data_px2

non_essential_sum = data_px2[['SiO2',"Al2O3","MnO","MgO","CaO", "Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<5]
m = data_px2[["FeO","MnO","MgO","CaO"]].sum(axis=1)
data_px2 = data_px2[(data_px2['Al2O3']<=14) & (m >=42) & (m <= 100)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Mag/Hem"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,


In [12]:
data_cleaned[data_cleaned["MINERAL"]=="CHROME-SPINEL"]

,SIO2(WT%),TIO2(WT%),AL2O3(WT%),CR2O3(WT%),FE2O3T(WT%),FE2O3(WT%),FEOT(WT%),FEO(WT%),MNO(WT%),MGO(WT%),CAO(WT%),NA2O(WT%),K2O(WT%),P2O5(WT%),MINERAL
CITATION,,,,,,,,,,,,,,,


In [13]:
andradite =data_cleaned[data_cleaned['MINERAL']=='ANDRADITE']

In [14]:
data_cleaned = data2.loc[~data2['FEO(WT%)'].isna(),:]
# data_px = data_cleaned.loc[(data_cleaned['MINERAL']=="ILMENITE"),:]
data_px = data_cleaned.copy()
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'Mineral']
mineral = data_px['Mineral']
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>99) & (total<101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data
data_px2

non_essential_sum = data_px2[['SiO2',"Al2O3","MnO","MgO","CaO", "Na2O", "K2O"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<5]
m = data_px2[["FeO","MnO","MgO","CaO"]].sum(axis=1)
data_px2 = data_px2[(data_px2['SiO2']<=37) & (data_px2['SiO2']>=38)  & (data_px2['FeO'] >=24) & (m <= 43)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Grt2"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,Mineral
CITATION,,,,,,,,,,,
